In [ ]:
!nvidia-smi


In [ ]:
%cd /kaggle/working
!rm -rf FullSubNet  
!git clone https://github.com/Audio-WestlakeU/FullSubNet.git
%cd FullSubNet

!pip install -q librosa tbb tensorboard joblib matplotlib pesq pystoi tqdm toml rich soundfile
!pip install -q --no-deps torch_complex
!pip install -q https://github.com/vBaiCai/python-pesq/archive/master.zip


In [ ]:
import torch
print("torch version:", torch.__version__)
print("torch cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Torch dang khong thay CUDA - restart session roi chay lai tu Buoc 0"

In [ ]:
import os

CLEAN_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN"
NOISE_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE"
TEST_DIR  = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST "

CKPT_PATH = "/kaggle/working/FullSubNet_EN.tar"   # sua neu ban de checkpoint noi khac

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.isdir(CLEAN_DIR), f"Khong tim thay: {CLEAN_DIR}"
assert os.path.isdir(NOISE_DIR), f"Khong tim thay: {NOISE_DIR}"
assert os.path.isdir(TEST_DIR), f"Khong tim thay: {TEST_DIR}"
print("OK, cac thu muc du lieu ton tai.")


In [ ]:
AUDIO_EXT = (".wav", ".flac")

def list_audio_files(folder, exts=AUDIO_EXT):
    files = []
    for root, _, names in os.walk(folder):
        for n in names:
            if n.lower().endswith(exts):
                files.append(os.path.join(root, n))
    return sorted(files)

clean_files = list_audio_files(CLEAN_DIR)
noise_files = list_audio_files(NOISE_DIR)
test_files  = list_audio_files(TEST_DIR)

print("So file clean:", len(clean_files))
print("So file noise:", len(noise_files))
print("So file test (noisy):", len(test_files))

assert len(clean_files) > 0, "Khong tim thay file audio nao trong CLEAN_DIR, kiem tra lai duong dan/dinh dang"
assert len(noise_files) > 0, "Khong tim thay file audio nao trong NOISE_DIR, kiem tra lai duong dan/dinh dang"


In [ ]:
import random, numpy as np, soundfile as sf

random.seed(0)
N_VAL = min(40, len(clean_files) // 20 + 1)  
val_clean_pool = random.sample(clean_files, min(N_VAL, len(clean_files)))
val_noise_pool = [random.choice(noise_files) for _ in val_clean_pool]

VAL_DIR = os.path.join(WORK_DIR, "no_reverb")
VAL_CLEAN_DIR = os.path.join(VAL_DIR, "clean")
VAL_NOISY_DIR = os.path.join(VAL_DIR, "noisy")
os.makedirs(VAL_CLEAN_DIR, exist_ok=True)
os.makedirs(VAL_NOISY_DIR, exist_ok=True)

TARGET_SR = 16000

def load_mono(path, target_sr=TARGET_SR):
    wav, sr = sf.read(path, always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        import librosa
        wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=target_sr)
    return wav.astype(np.float32)

def mix_snr(clean_wav, noise_wav, snr_db=10):
    if len(noise_wav) < len(clean_wav):
        reps = len(clean_wav) // len(noise_wav) + 1
        noise_wav = np.tile(noise_wav, reps)
    noise_wav = noise_wav[: len(clean_wav)]
    clean_power = np.mean(clean_wav ** 2) + 1e-8
    noise_power = np.mean(noise_wav ** 2) + 1e-8
    scale = np.sqrt(clean_power / (noise_power * (10 ** (snr_db / 10))))
    noisy = clean_wav + noise_wav * scale
    return noisy

for i, (c_path, n_path) in enumerate(zip(val_clean_pool, val_noise_pool)):
    clean_wav = load_mono(c_path)
    noise_wav = load_mono(n_path)
    snr_db = random.uniform(0, 15)
    noisy_wav = mix_snr(clean_wav, noise_wav, snr_db)
    sf.write(os.path.join(VAL_CLEAN_DIR, f"clean_fileid_{i}.wav"), clean_wav, TARGET_SR)
    sf.write(os.path.join(VAL_NOISY_DIR, f"noisy_fileid_{i}.wav"), noisy_wav, TARGET_SR)

print(f"Da tao {len(val_clean_pool)} cap validation tai {VAL_DIR}")


val_clean_set = set(val_clean_pool)
clean_files_train = [f for f in clean_files if f not in val_clean_set]
noise_files_train = noise_files  

print("So file clean con lai cho train:", len(clean_files_train))


In [ ]:
CLEAN_TXT = os.path.join(WORK_DIR, "clean.txt")
NOISE_TXT = os.path.join(WORK_DIR, "noise.txt")

with open(CLEAN_TXT, "w") as f:
    f.write("\n".join(clean_files_train))
with open(NOISE_TXT, "w") as f:
    f.write("\n".join(noise_files_train))

print("Da ghi:", CLEAN_TXT, NOISE_TXT)


In [ ]:
CKPT_URL = "https://github.com/hkha0801-sketch/PESEM-VS/raw/main/checkpoints/FullSubNet_EN.tar"
CKPT_PATH = "/kaggle/working/FullSubNet_EN.tar"

!wget -q --show-progress -O "{CKPT_PATH}" "{CKPT_URL}"

import os
size_mb = os.path.getsize(CKPT_PATH) / (1024 * 1024)
print(f"Da tai checkpoint ve: {CKPT_PATH} ({size_mb:.1f} MB)")
assert size_mb > 1, "File qua nho, co the wget da tai ve trang HTML loi thay vi file tar that su - kiem tra lai CKPT_URL"

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/FullSubNet")
sys.path.insert(0, "/kaggle/working/FullSubNet/recipes/dns_interspeech_2020")

import torch
from fullsubnet.model import Model

MODEL_ARGS = dict(
    sb_num_neighbors=15,
    fb_num_neighbors=0,
    num_freqs=257,
    look_ahead=2,
    sequence_model="LSTM",
    fb_output_activate_function="ReLU",
    sb_output_activate_function=False,
    fb_model_hidden_size=512,
    sb_model_hidden_size=384,
    weight_init=False,
    norm_type="offline_laplace_norm",
    num_groups_in_drop_band=2,
)

ckpt_raw = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state_dict = ckpt_raw.get("model", ckpt_raw) if isinstance(ckpt_raw, dict) else ckpt_raw

model = Model(**MODEL_ARGS)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
print("Neu 2 danh sach tren rong -> checkpoint khop hoan toan voi kien truc model.")
print("Neu khong rong -> kiem tra lai MODEL_ARGS co dung voi checkpoint goc khong truoc khi train tiep.")

In [ ]:
CLEAN_CKPT_PATH = os.path.join(WORK_DIR, "pretrained_for_finetune.tar")
torch.save({"epoch": 0, "model": model.state_dict()}, CLEAN_CKPT_PATH)
print("Da luu checkpoint sach tai:", CLEAN_CKPT_PATH)


In [ ]:
import toml

TOML_PATH = "/kaggle/working/FullSubNet/recipes/dns_interspeech_2020/fullsubnet/train.toml"
cfg = toml.load(TOML_PATH)

SAVE_DIR = os.path.join(WORK_DIR, "Experiments", "FullSubNet")
cfg["meta"]["save_dir"] = SAVE_DIR

cfg["train_dataset"]["args"]["clean_dataset"] = CLEAN_TXT
cfg["train_dataset"]["args"]["noise_dataset"] = NOISE_TXT
cfg["train_dataset"]["args"]["reverb_proportion"] = 0.0  # khong co RIR dataset
cfg["train_dataset"]["args"]["num_workers"] = 2
cfg["train_dataset"]["args"]["sr"] = TARGET_SR

cfg["train_dataset"]["dataloader"]["batch_size"] = 8       # giam neu bi OOM tren GPU Kaggle
cfg["train_dataset"]["dataloader"]["num_workers"] = 2

cfg["validation_dataset"]["args"]["dataset_dir_list"] = [VAL_DIR + "/"]
cfg["validation_dataset"]["args"]["sr"] = TARGET_SR

cfg["model"]["args"] = MODEL_ARGS

cfg["trainer"]["train"]["epochs"] = 20
cfg["trainer"]["train"]["save_checkpoint_interval"] = 1
cfg["trainer"]["validation"]["validation_interval"] = 1
cfg["trainer"]["visualization"]["num_workers"] = 2

with open(TOML_PATH, "w") as f:
    toml.dump(cfg, f)

print("Da cap nhat:", TOML_PATH)
print(toml.dumps(cfg))


In [ ]:
import re

trainer_file = "/kaggle/working/FullSubNet/audio_zen/trainer/base_trainer.py"
with open(trainer_file, "r") as f:
    src = f.read()

def add_weights_only(match):
    call = match.group(0)
    if "weights_only" in call:
        return call
    return call[:-1] + ", weights_only=False)"

new_src = re.sub(r"torch\.load\(.*\)", add_weights_only, src)

if new_src != src:
    with open(trainer_file, "w") as f:
        f.write(new_src)
    print("Da patch:", trainer_file)
else:
    print("Khong co gi de patch (co the da patch roi, hoac khong tim thay torch.load o dang mong doi).")

# Kiem tra lai file co con loi cu phap khong, fail som thay vi de train.py crash sau
import py_compile
try:
    py_compile.compile(trainer_file, doraise=True)
    print("OK: base_trainer.py hop le ve cu phap sau khi patch.")
except py_compile.PyCompileError as e:
    raise RuntimeError(f"base_trainer.py van con loi cu phap sau khi patch: {e}")


In [ ]:
RIR_TXT = os.path.join(WORK_DIR, "rir.txt")
with open(RIR_TXT, "w") as f:
    f.write("\n".join(noise_files_train[:5]))
print("Da tao file rir gia (khong dung thuc su):", RIR_TXT)

In [ ]:
import toml

TOML_PATH = "/kaggle/working/FullSubNet/recipes/dns_interspeech_2020/fullsubnet/train.toml"
cfg = toml.load(TOML_PATH)

SAVE_DIR = os.path.join(WORK_DIR, "Experiments", "FullSubNet")
cfg["meta"]["save_dir"] = SAVE_DIR

cfg["train_dataset"]["args"]["clean_dataset"] = CLEAN_TXT
cfg["train_dataset"]["args"]["noise_dataset"] = NOISE_TXT
cfg["train_dataset"]["args"]["rir_dataset"] = RIR_TXT
cfg["train_dataset"]["args"]["reverb_proportion"] = 0.0
cfg["train_dataset"]["args"]["num_workers"] = 2
cfg["train_dataset"]["args"]["sr"] = TARGET_SR

cfg["train_dataset"]["dataloader"]["batch_size"] = 8
cfg["train_dataset"]["dataloader"]["num_workers"] = 2

cfg["validation_dataset"]["args"]["dataset_dir_list"] = [VAL_DIR + "/"]
cfg["validation_dataset"]["args"]["sr"] = TARGET_SR

# CLI flag -P bi bug trong ban train.py hien tai, khong duoc ghi vao config dict
# (gay KeyError: 'preloaded_model_path'), nen ghi thang key nay vao file toml
cfg["preloaded_model_path"] = CKPT_PATH

cfg["model"]["args"] = dict(
    sb_num_neighbors=15, fb_num_neighbors=0, num_freqs=257, look_ahead=2,
    sequence_model="LSTM", fb_output_activate_function="ReLU",
    sb_output_activate_function=False, fb_model_hidden_size=512,
    sb_model_hidden_size=384, weight_init=False,
    norm_type="offline_laplace_norm", num_groups_in_drop_band=2,
)

cfg["trainer"]["train"]["epochs"] = 20
cfg["trainer"]["train"]["save_checkpoint_interval"] = 1
cfg["trainer"]["validation"]["validation_interval"] = 1
cfg["trainer"]["visualization"]["num_workers"] = 2

with open(TOML_PATH, "w") as f:
    toml.dump(cfg, f)

print("Da cap nhat:", TOML_PATH)
print(toml.dumps(cfg))

In [ ]:
%cd /kaggle/working/FullSubNet/recipes/dns_interspeech_2020
!torchrun --standalone --nnodes=1 --nproc_per_node=1 train.py -C fullsubnet/train.toml -P /kaggle/working/FullSubNet_EN.tar

In [ ]:
import glob

exp_dirs = glob.glob(os.path.join(SAVE_DIR, "*"))
print("Cac thu muc experiment tim thay:", exp_dirs)


In [ ]:
EXP_DIR = exp_dirs[0] if exp_dirs else None
assert EXP_DIR is not None, "Chua co thu muc experiment, hay chay lai cell train o Buoc 7 truoc"

CKPT_DIR = os.path.join(EXP_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

import shutil
target_ckpt = os.path.join(CKPT_DIR, "latest_model.tar")
shutil.copy(CLEAN_CKPT_PATH, target_ckpt)
print("Da copy checkpoint sach vao:", target_ckpt)


In [ ]:
%cd /kaggle/working/FullSubNet/recipes/dns_interspeech_2020
!torchrun --standalone --nnodes=1 --nproc_per_node=1 train.py -C fullsubnet/train.toml -R

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {SAVE_DIR}


In [ ]:
INFER_TOML_PATH = "/kaggle/working/FullSubNet/recipes/dns_interspeech_2020/fullsubnet/inference.toml"
infer_cfg = toml.load(INFER_TOML_PATH)
print(toml.dumps(infer_cfg))
print("---")
print("Kiem tra key duong dan du lieu (thuong la 'dataset_dir' hoac tuong tu) va sua thanh:", TEST_DIR)


In [ ]:
BEST_CKPT = os.path.join(CKPT_DIR, "best_model.tar")
OUTPUT_DIR = "/kaggle/working/enhanced_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

%cd /kaggle/working/FullSubNet/recipes/dns_interspeech_2020
!python inference.py -C fullsubnet/inference.toml -M {BEST_CKPT} -O {OUTPUT_DIR}
